# Concurrent Vendor Risk Assessment

| Field | Value |
| --- | --- |
| Scenario id | `concurrent-vendor-risk-assessment` |
| Pattern | `concurrent` |
| API | `Responses API` |

**Learning goal:** Learn how a Responses API endpoint can fan out one vendor intake question to independent enterprise risk reviewers.

> Use Responses plus concurrent orchestration when independent departments can assess the same vendor in parallel for a single requester.


### Runtime configuration

**Supporting code.** Reads the Ollama model/host from the environment and creates the shared `MCP_TOOL_FUNCTIONS` registry that later cells populate and agents read from.


In [ ]:
import os

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:14b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")

# Domain tools register themselves here; every agent looks up its granted
# tools by name, so this registry is the one piece of shared runtime state.
MCP_TOOL_FUNCTIONS: dict[str, object] = {}

print(f"Ollama target: {OLLAMA_HOST} / {OLLAMA_MODEL}")


### Notebook styling

**Supporting code.** Loads the Aptos look and the per-agent accent colors the render helpers reuse. Pure presentation -- no Agent Framework surface here.


In [ ]:
from IPython.display import HTML, Markdown, display


_AGENT_COLORS = ("#3868c8", "#b0530f", "#2f7d4f", "#7d3f98", "#a3374b", "#0f7d8a", "#8a6d0f", "#54596b")

_APTOS_STYLE = """
<style>
:root { --jp-content-font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif; }
.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .rendered_html, .jp-OutputArea-output {
    font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif;
    line-height: 1.55;
}
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
    font-family: 'Aptos Display', 'Aptos', 'Segoe UI', sans-serif;
    font-weight: 600;
}
.maf-callout {
    border-left: 4px solid #3868c8; border-radius: 6px; padding: 0.6em 0.9em;
    margin: 0.6em 0; background: rgba(56, 104, 200, 0.08);
}
.maf-roster { display: flex; flex-wrap: wrap; gap: 0.6em; margin: 0.4em 0; }
.maf-card {
    border: 1px solid rgba(128, 128, 128, 0.35); border-radius: 8px;
    padding: 0.55em 0.8em; min-width: 14em; max-width: 24em; flex: 1;
}
.maf-card b { display: block; margin-bottom: 0.15em; }
.maf-card small { opacity: 0.75; }
.maf-chip {
    display: inline-block; border-radius: 999px; padding: 0 0.6em; margin: 0.2em 0.2em 0 0;
    font-size: 0.78em; border: 1px solid rgba(128, 128, 128, 0.4);
}
.maf-turn {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.45em 0.8em; margin: 0.45em 0; background: rgba(128, 128, 128, 0.07);
    white-space: pre-wrap;
}
.maf-turn b { color: var(--maf-agent, inherit); }
.maf-turn-label {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.3em 0.7em; margin: 0.7em 0 0.15em; background: rgba(128, 128, 128, 0.09);
}
.maf-turn-label b { color: var(--maf-agent, inherit); }
</style>
"""


def apply_notebook_style() -> str:
    display(HTML(_APTOS_STYLE))
    return _APTOS_STYLE


apply_notebook_style()


### Rendering helpers

**Supporting code.** `render_roster` and `render_transcript` turn scenario specs and workflow output into readable HTML and markdown. Glue for the notebook, not framework API.


In [ ]:
import re as _re


def _escape_html(value) -> str:
    import html as _html

    return _html.escape(str(value))


def agent_color(name: str) -> str:
    """Deterministic per-agent accent color, stable across cells and runs."""

    return _AGENT_COLORS[sum(ord(ch) for ch in name) % len(_AGENT_COLORS)]


def render_callout(text: str) -> None:
    display(HTML("<div class='maf-callout'>" + _escape_html(text) + "</div>"))


def render_roster(scenario) -> None:
    """Render the agent roster as color-accented cards with tool chips."""

    cards = []
    for spec in scenario.agents:
        chips = "".join(
            "<span class='maf-chip'>" + _escape_html(tool) + "</span>" for tool in spec.mcp_tools
        ) or "<span class='maf-chip'>instructions only</span>"
        cards.append(
            "<div class='maf-card' style='border-top: 3px solid " + agent_color(spec.name) + "'>"
            + "<b>" + _escape_html(spec.name) + "</b>"
            + "<small>" + _escape_html(spec.description) + "</small>"
            + "<div>" + chips + "</div></div>"
        )
    display(HTML("<div class='maf-roster'>" + "".join(cards) + "</div>"))


_TURN_LABEL = _re.compile(r"^\[([A-Za-z0-9_]+)\]\s*", _re.MULTILINE)


def render_transcript(text: str) -> None:
    """Render workflow output as color-coded per-agent turns.

    Each turn's body is emitted as a ``text/markdown`` output (via
    ``Markdown``) so Jupyter renders the agent's markdown, while the
    per-agent accent color rides on an HTML label bar above the body.
    """

    pieces = _TURN_LABEL.split(text)
    preamble = pieces[0].strip()
    labeled = list(zip(pieces[1::2], pieces[2::2]))
    if not preamble and not labeled:
        display(Markdown(text))
        return
    if preamble:
        display(Markdown(preamble))
    for label, body in labeled:
        display(HTML(
            "<div class='maf-turn-label' style='--maf-agent: " + agent_color(label) + "'>"
            + "<b>" + _escape_html(label) + "</b></div>"
        ))
        display(Markdown(body.strip()))


## Pattern: Concurrent

Concurrent orchestration fans one request out to several specialists. They work independently, then a code-defined fan-in executor labels and combines their findings. Scenarios with a designated synthesizer hold that agent out of the fan-out and run it after fan-in, so the agent that combines the perspectives actually sees them.

Best fit: independent reviews where parallel perspectives are more valuable than turn-taking.

## API Shape

**Responses API -- startup-selected scenario shape.** The scenario and orchestration pattern are wired in at server start. Each client request uses the standard OpenAI-compatible `/responses` body -- a plain chat-style input. The client never specifies which agents run; the server owns the orchestration entirely.

This is a starter scenario. The domain is intentionally lightweight so the orchestration mechanics are easy to trace before the enterprise and quote-to-cash notebooks layer in MCP tool calls and richer context.

## Pattern Anatomy

| Dimension | Detail |
| --- | --- |
| Control flow | Parallel lanes receive the same input and run independently; an optional synthesizer runs after fan-in. |
| Coordination | The graph fans out work, aggregates labelled outputs, and can forward them to a synthesis agent. |
| Output | Each lane contributes a labelled perspective; a synthesizer combines them when declared. |
| Best when | Use when independent reviews can happen in parallel. |

## Instruction-Led LLM Agents

| Agent | Role | Capabilities |
| --- | --- | --- |
| `SecurityRiskAgent` | Reviews security controls and access exposure. | Instructions only |
| `PrivacyRiskAgent` | Reviews personal-data and customer-data handling. | Instructions only |
| `LegalRiskAgent` | Reviews contract and liability posture. | Instructions only |
| `FinanceRiskAgent` | Reviews cost, budget, and renewal exposure. | Instructions only |
| `OperationsRiskAgent` | Reviews supportability and integration readiness. | Instructions only |

> **Instructor note:** Each row is an LLM-backed agent with role instructions.
> Most agents rely on instructions alone; enterprise and quote-to-cash agents may
> also call domain tools for grounded context.


### Scenario schema

**Supporting code.** Plain frozen dataclasses -- `AgentSpec` and `ScenarioSpec` -- that mirror the scenario JSON. Not framework types; just the shape every later cell reads.


In [ ]:
from dataclasses import dataclass
from typing import Any, Sequence


@dataclass(frozen=True)
class AgentSpec:
    name: str
    description: str
    instructions: str
    mcp_tools: tuple[str, ...] = ()
    mcp_server: str = "enterprise_context"
    route_keywords: tuple[str, ...] = ()
    a2a_url: str | None = None


@dataclass(frozen=True)
class ScenarioSpec:
    id: str
    pattern: str
    title: str
    learning_goal: str
    when_to_use: str
    sample_input: str
    agents: tuple[AgentSpec, ...]
    handoff_finisher: str | None = None
    concurrent_synthesizer: str | None = None
    termination_phrases: tuple[str, ...] = ()


### Load the scenario

**Supporting code.** Hydrates the embedded JSON into the `SCENARIO` object this notebook drives. Data plumbing, no Agent Framework surface.


In [ ]:
import json


SCENARIO_DATA = json.loads(
    r"""
{
  "id": "concurrent-vendor-risk-assessment",
  "pattern": "concurrent",
  "title": "Concurrent Vendor Risk Assessment",
  "learning_goal": "Learn how a Responses API endpoint can fan out one vendor intake question to independent enterprise risk reviewers.",
  "when_to_use": "Use Responses plus concurrent orchestration when independent departments can assess the same vendor in parallel for a single requester.",
  "sample_input": "Assess whether we should approve a new analytics vendor that will process customer usage data and integrate with our warehouse. Constraints: the annual budget cap is 150k USD, a decision is needed within two weeks, and the data science team wants API access in the first month.",
  "handoff_finisher": null,
  "concurrent_synthesizer": null,
  "termination_phrases": [],
  "agents": [
    {
      "name": "SecurityRiskAgent",
      "description": "Reviews security controls and access exposure.",
      "instructions": "Assess identity, encryption, data access, vulnerability management, and incident-response risk.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "PrivacyRiskAgent",
      "description": "Reviews personal-data and customer-data handling.",
      "instructions": "Assess data minimization, retention, subprocessors, regional transfer, and privacy notice implications.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "LegalRiskAgent",
      "description": "Reviews contract and liability posture.",
      "instructions": "Assess indemnity, limitation of liability, audit rights, termination, and compliance clauses.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "FinanceRiskAgent",
      "description": "Reviews cost, budget, and renewal exposure.",
      "instructions": "Assess pricing model, renewal terms, hidden costs, budget fit, and procurement risk.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "OperationsRiskAgent",
      "description": "Reviews supportability and integration readiness.",
      "instructions": "Assess integration effort, service reliability, support model, monitoring, and operational fallback.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    }
  ]
}
    """
)
AGENTS = tuple(
    AgentSpec(
        name=item["name"],
        description=item["description"],
        instructions=item["instructions"],
        mcp_tools=tuple(item.get("mcp_tools", [])),
        mcp_server=item.get("mcp_server", "enterprise_context"),
        route_keywords=tuple(item.get("route_keywords", [])),
        a2a_url=item.get("a2a_url"),
    )
    for item in SCENARIO_DATA["agents"]
)
SCENARIO = ScenarioSpec(
    id=SCENARIO_DATA["id"],
    pattern=SCENARIO_DATA["pattern"],
    title=SCENARIO_DATA["title"],
    learning_goal=SCENARIO_DATA["learning_goal"],
    when_to_use=SCENARIO_DATA["when_to_use"],
    sample_input=SCENARIO_DATA["sample_input"],
    agents=AGENTS,
    handoff_finisher=SCENARIO_DATA.get("handoff_finisher"),
    concurrent_synthesizer=SCENARIO_DATA.get("concurrent_synthesizer"),
    termination_phrases=tuple(SCENARIO_DATA.get("termination_phrases", [])),
)

print(f"Loaded {SCENARIO.title} with {len(SCENARIO.agents)} agents.")


### Inspection helpers

**Supporting code.** `agent_capability_map`, `mcp_tool_context`, and `tools_for_agent` let you read the roster and resolve each agent's granted tools by name -- supporting utilities used later by `make_agent`.


In [ ]:
def tools_for_agent(spec: AgentSpec) -> list[object]:
    tools: list[object] = []
    for tool_name in spec.mcp_tools:
        try:
            tools.append(MCP_TOOL_FUNCTIONS[tool_name])
        except KeyError as exc:
            raise ValueError(f"Missing inlined tool '{tool_name}' for {spec.name}.") from exc
    return tools


def scenario_summary(scenario: ScenarioSpec) -> dict[str, str]:
    return {
        "id": scenario.id,
        "title": scenario.title,
        "pattern": scenario.pattern,
        "learning_goal": scenario.learning_goal,
        "when_to_use": scenario.when_to_use,
        "sample": getattr(scenario, "sample_input"),
    }


def agent_capability_map(scenario: ScenarioSpec) -> list[dict[str, Any]]:
    return [
        {
            "agent": spec.name,
            "description": spec.description,
            "instructions": spec.instructions,
            "mcp_tools": list(spec.mcp_tools),
            "mcp_server": spec.mcp_server if spec.mcp_tools else None,
        }
        for spec in scenario.agents
    ]


def mcp_tool_context(scenario: ScenarioSpec) -> dict[str, Any]:
    tools_by_agent = {spec.name: list(spec.mcp_tools) for spec in scenario.agents if spec.mcp_tools}
    all_tools_used = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    return {
        "uses_mcp": bool(all_tools_used),
        "tools_by_agent": tools_by_agent,
        "all_tools_used": all_tools_used,
    }


### Sample prompt and budget

**Supporting code.** Fixes the token budget and the exact `SAMPLE_PROMPT` the live run will send, then prints the roster so you can see who is on the team before any orchestration.


In [ ]:
import json


MAX_TOKENS = 500 if SCENARIO.pattern == "magentic" else 250
RESPONSES_PAYLOAD = {"input": SCENARIO.sample_input, "stream": False}
SAMPLE_PROMPT = SCENARIO.sample_input

render_roster(SCENARIO)
print(json.dumps(scenario_summary(SCENARIO), indent=2))
print(json.dumps(agent_capability_map(SCENARIO), indent=2))
if mcp_tool_context(SCENARIO)["uses_mcp"]:
    print(json.dumps(mcp_tool_context(SCENARIO), indent=2))


### Ollama configuration

**Supporting code.** Frozen `OllamaAgentConfig` plus env-driven defaults. This is local-runtime plumbing, independent of any Agent Framework class.


In [ ]:
from dataclasses import dataclass
from typing import Any

from agent_framework.ollama import OllamaChatClient


DEFAULT_OLLAMA_TEMPERATURE = 0.0
DEFAULT_OLLAMA_NUM_CTX = 8192
DEFAULT_OLLAMA_KEEP_ALIVE = "10m"
DEFAULT_OLLAMA_THINK = False

# Ollama's chat endpoint rejects a few OpenAI-style options; we strip these later.
_UNSUPPORTED_OLLAMA_CHAT_OPTIONS = {
    "allow_multiple_tool_calls",
    "conversation_id",
    "logit_bias",
    "metadata",
    "store",
    "user",
}


@dataclass(frozen=True)
class OllamaAgentConfig:
    model: str
    host: str
    temperature: float
    num_ctx: int
    max_tokens: int
    keep_alive: str
    think: bool

    def default_options(self) -> dict[str, Any]:
        return {
            "temperature": self.temperature,
            "num_ctx": self.num_ctx,
            "max_tokens": self.max_tokens,
            "keep_alive": self.keep_alive,
            "think": self.think,
        }


def parse_env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true or false.")


def build_ollama_config(
    *,
    model: str | None = None,
    host: str | None = None,
    temperature: float | None = None,
    num_ctx: int | None = None,
    max_tokens: int | None = None,
    keep_alive: str | None = None,
    think: bool | None = None,
) -> OllamaAgentConfig:
    return OllamaAgentConfig(
        model=model or os.getenv("OLLAMA_MODEL") or "qwen3:14b",
        host=host or os.getenv("OLLAMA_HOST") or "http://localhost:11434",
        temperature=temperature
        if temperature is not None
        else float(os.getenv("OLLAMA_TEMPERATURE", str(DEFAULT_OLLAMA_TEMPERATURE))),
        num_ctx=num_ctx if num_ctx is not None else int(os.getenv("OLLAMA_NUM_CTX", str(DEFAULT_OLLAMA_NUM_CTX))),
        max_tokens=max_tokens if max_tokens is not None else int(os.getenv("OLLAMA_MAX_TOKENS", "500")),
        keep_alive=keep_alive or os.getenv("OLLAMA_KEEP_ALIVE") or DEFAULT_OLLAMA_KEEP_ALIVE,
        think=think if think is not None else parse_env_bool("OLLAMA_THINK", DEFAULT_OLLAMA_THINK),
    )


### Chat-client shim

**Supporting code.** A thin `OllamaChatClient` subclass that filters out chat options the local Ollama server rejects -- an adapter, not framework surface.


In [ ]:
class ScenarioOllamaChatClient(OllamaChatClient):
    """OllamaChatClient that drops chat options the local Ollama server rejects."""

    def _prepare_options(self, messages: Any, options: Any) -> dict[str, Any]:
        filtered = {
            key: value
            for key, value in dict(options).items()
            if key not in _UNSUPPORTED_OLLAMA_CHAT_OPTIONS
        }
        return super()._prepare_options(messages, filtered)


### make_agent

**Agent Framework primitive.** The factory: `client.as_agent(...)` turns a chat client + role instructions + granted tools into an agent (or an `A2AAgent` for remote peers). This is the Agent Framework's core agent-construction call.


In [ ]:
def make_agent(spec: Any, *, config: OllamaAgentConfig | None = None) -> Any:
    if spec.a2a_url:
        from agent_framework.a2a import A2AAgent

        url = spec.a2a_url
        if not url.startswith("http"):
            url = os.getenv("A2A_PARTNER_BASE_URL", "http://localhost:8765").rstrip("/") + url
        return A2AAgent(name=spec.name, description=spec.description, url=url)

    resolved = config or build_ollama_config()
    instructions = f"You are {spec.name}. {spec.instructions}"
    tools = tools_for_agent(spec)
    return ScenarioOllamaChatClient(host=resolved.host, model=resolved.model).as_agent(
        name=spec.name,
        description=spec.description,
        instructions=instructions,
        tools=tools or None,
        default_options=resolved.default_options(),
        require_per_service_call_history_persistence=True,
    )


print("Agent factory ready: make_agent(spec) creates an instruction-led Ollama agent "
      "with its granted tools attached.")


### Framework imports and message helpers

**Agent Framework primitive.** Imports the workflow primitives (`Executor`, `WorkflowBuilder`, `WorkflowContext`, `AgentExecutor`, `handler`, ...) and wraps text into an `AgentExecutorRequest`. This is the Agent Framework surface every pattern builds on.


In [ ]:
import re
from typing import Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)


# State keys shared across executors: the running transcript, and the stopwords
# the handoff router strips when it derives routing keywords from agent names.
_TRANSCRIPT_KEY = "transcript"
_STOPWORDS = {"agent", "specialist", "the", "and", "for", "with", "that", "from", "into"}


def make_request(text: str) -> AgentExecutorRequest:
    return AgentExecutorRequest(messages=[Message(role="user", contents=[text])])


def response_text(response: AgentExecutorResponse) -> str:
    agent_response = getattr(response, "agent_response", None)
    return (getattr(agent_response, "text", None) or "").strip()


### Transcript state

**Supporting code.** A helper that appends a labelled turn to workflow state via `ctx.get_state`/`set_state`. Bookkeeping the executors reuse -- not framework API itself.


In [ ]:
def _append_transcript(ctx: WorkflowContext[Any], author: str, text: str) -> list[str]:
    transcript = list(ctx.get_state(_TRANSCRIPT_KEY) or [])
    transcript.append(f"[{author}] {text}")
    ctx.set_state(_TRANSCRIPT_KEY, transcript)
    return transcript


### Your first executor

**Agent Framework primitive.** `PromptDispatchExecutor` subclasses `Executor` and marks its method with `@handler`. It seeds state and `send_message`s the first request -- the entry node of every graph here.


In [ ]:
class PromptDispatchExecutor(Executor):
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        ctx.set_state("prompt", prompt)
        ctx.set_state(_TRANSCRIPT_KEY, [])
        await ctx.send_message(make_request(prompt))


### Agents as workflow nodes

**Agent Framework primitive.** `_agent_executor` wraps an agent in an `AgentExecutor` so it can sit in the graph. This is the bridge from a standalone agent to a workflow node.


In [ ]:
def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _agents_for(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> list[Any]:
    return [make_agent(spec, config=config) for spec in scenario.agents]


def _agent_executor(spec_index: int, scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> AgentExecutor:
    spec = scenario.agents[spec_index]
    return AgentExecutor(make_agent(spec, config=config), id=_slug(spec.name))


print("Workflow plumbing ready: dispatch executor, shared transcript state, and "
      "request/response helpers.")


### Attribute each parallel result

**Supporting code.** Fan-in delivers responses in nondeterministic order, so this helper pairs each response back to its agent name (by executor id, with a positional fallback). Pure bookkeeping.


In [ ]:
def _labelled_responses(responses: list, agent_names: list) -> list:
    """Pair fan-in responses with agent names by executor_id, position fallback."""

    names_by_slug = {_slug(name): name for name in agent_names}
    labelled = []
    for index, response in enumerate(responses):
        name = names_by_slug.get(getattr(response, "executor_id", None) or "")
        if name is None:
            name = agent_names[index] if index < len(agent_names) else f"agent{index + 1}"
        labelled.append((name, response_text(response)))
    return labelled


### ConcurrentAggregatorExecutor: fan-in and combine

**Agent Framework primitive.** The fan-in `Executor`. Its handler receives the whole `list` of specialist responses at once and `yield_output`s the labelled combination -- the terminal node when no synthesizer is declared.


In [ ]:
class ConcurrentAggregatorExecutor(Executor):
    def __init__(self, id: str, *, agent_names: list[str]) -> None:
        super().__init__(id=id)
        self._agent_names = agent_names

    @handler
    async def aggregate(self, responses: list[AgentExecutorResponse], ctx: WorkflowContext[Never, str]) -> None:
        labelled = _labelled_responses(responses, self._agent_names)
        await ctx.yield_output("\n\n".join(f"[{name}]\n{text}" for name, text in labelled))


### ConcurrentSynthesisGateExecutor: forward to a synthesizer

**Agent Framework primitive.** When the scenario names a synthesizer, this fan-in `Executor` instead forwards the labelled findings onward as a new request, so a final agent can reconcile them.


In [ ]:
class ConcurrentSynthesisGateExecutor(Executor):
    def __init__(self, id: str, *, agent_names: list[str]) -> None:
        super().__init__(id=id)
        self._agent_names = agent_names

    @handler
    async def gate(self, responses: list[AgentExecutorResponse], ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        for name, text in _labelled_responses(responses, self._agent_names):
            _append_transcript(ctx, name, text)
        prompt = ctx.get_state("prompt") or ""
        carried = "\n".join(ctx.get_state(_TRANSCRIPT_KEY) or [])
        await ctx.send_message(
            make_request(
                f"You are the synthesis stage.\nOriginal request:\n{prompt}\n\n"
                f"Independent specialist findings:\n{carried}\n\n"
                "Combine these findings into the final deliverable."
            )
        )


### Terminal output executor

**Agent Framework primitive.** The terminal `Executor` that yields the joined transcript. It only runs on the synthesizer path (after the synthesis gate); the aggregator path ends at the aggregator itself.


In [ ]:
class SequentialOutputExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def finish(self, response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        await ctx.yield_output("\n\n".join(transcript))


### Preview fan-in labelling

**Supporting code.** An offline check -- no model call -- of how each parallel lane is labelled before aggregation.


In [ ]:
# Demo (offline): how fan-in labels each parallel finding before aggregation.
_parallel = [spec.name for spec in SCENARIO.agents if spec.name != SCENARIO.concurrent_synthesizer][:3]
print("\n\n".join(f"[{name}]\n{name} would report its independent finding here." for name in _parallel))


### Wire fan-out and fan-in with WorkflowBuilder

**Agent Framework primitive.** `add_fan_out_edges` sends the request to every lane at once and `add_fan_in_edges` collects them. With a synthesizer, fan-in targets the synthesis gate and one more agent runs before output; without one, it targets the aggregator directly.


In [ ]:
def build_concurrent_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    synthesizer_name = scenario.concurrent_synthesizer
    parallel = [i for i in range(len(scenario.agents)) if scenario.agents[i].name != synthesizer_name]
    agents = [_agent_executor(i, scenario, config=config) for i in parallel]
    parallel_names = [scenario.agents[i].name for i in parallel]
    dispatch = PromptDispatchExecutor(id="dispatch")
    if synthesizer_name is None:
        aggregator = ConcurrentAggregatorExecutor(id="aggregator", agent_names=parallel_names)
        builder = WorkflowBuilder(start_executor=dispatch, output_from=[aggregator])
        builder.add_fan_out_edges(dispatch, agents)
        builder.add_fan_in_edges(agents, aggregator)
        return builder.build()
    synthesizer_index = next(
        i for i in range(len(scenario.agents)) if scenario.agents[i].name == synthesizer_name
    )
    synthesizer = _agent_executor(synthesizer_index, scenario, config=config)
    gate = ConcurrentSynthesisGateExecutor(id="synthesis_gate", agent_names=parallel_names)
    output = SequentialOutputExecutor(id="final_output", stage_name=synthesizer_name)
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_fan_out_edges(dispatch, agents)
    builder.add_fan_in_edges(agents, gate)
    builder.add_edge(gate, synthesizer)
    builder.add_edge(synthesizer, output)
    return builder.build()


### Compile and build

**Supporting code.** `build_workflow` resolves the Ollama config and calls the builder above, then `build()` compiles the runnable workflow. The wrapper is glue; `build()` is the framework call.


In [ ]:
def build_workflow(
    scenario: ScenarioSpec = SCENARIO,
    *,
    max_tokens: int | None = None,
    **config_overrides: Any,
) -> Any:
    config = build_ollama_config(max_tokens=max_tokens or MAX_TOKENS, **config_overrides)
    return build_concurrent_workflow(scenario, config=config)


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
print(
    f"Built the {SCENARIO.pattern} workflow over {len(SCENARIO.agents)} agents: "
    + type(workflow).__name__
)


## Flow Diagram

The diagram below shows a fan-out to 5 specialists and a labelled fan-in aggregation attached to the Responses API /responses.
Solid arrows are orchestration edges. Dashed arrows (`-.->`) are tool calls.
Domain tool nodes use a stadium shape.


In [ ]:
import base64
import html
from dataclasses import dataclass

from IPython.display import HTML, display


@dataclass(frozen=True)
class ScenarioFlowDiagram:
    title: str
    mermaid: str
    image_url: str


def scenario_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _concurrent_diagram(scenario, api_boundary="Responses API /responses", input_label="OpenAI-style input")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_scenario_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = scenario_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram



def _concurrent_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    synthesizer = next(
        (agent for agent in scenario.agents if agent.name == scenario.concurrent_synthesizer), None
    )
    parallel = [agent for agent in scenario.agents if agent is not synthesizer]
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append("    orchestrator --> fanout{Fan out same request}")
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(parallel, start=1):
        node = f"agent{index}"
        lines.append(f"    fanout --> {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> aggregate{{Aggregate findings}}")
        pairs.append((agent, node))
    if synthesizer is None:
        lines.append("    aggregate --> output[Responses output]")
    else:
        lines.append(f"    aggregate --> synthesizer[{_label(synthesizer.name)}]")
        lines.append("    synthesizer --> output[Responses output]")
        pairs.append((synthesizer, "synthesizer"))
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)



def _header(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> list[str]:
    return [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label(input_label)}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
    ]


def _mcp_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in agent.mcp_tools:
            lines.append(f"    {node_id} -.->|mcp tool| tool_{tool}([{_label(tool)}])")
    return lines


def quote_to_cash_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _quote_to_cash_source(scenario, api_boundary="Responses API /responses")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Quote-To-Cash Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_quote_to_cash_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = quote_to_cash_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _quote_to_cash_source(scenario: ScenarioSpec, *, api_boundary: str) -> str:
    names = {agent.name for agent in scenario.agents}

    def node(role: str) -> str:
        return role if role in names else next(iter(names))

    lines = [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label('Quote request begins in CRM')}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
        f"    orchestrator --> crm[{_label('CRM system')}]",
        f"    crm -->|wave 1| trigger[{_label(node('QuoteTriggerAgent'))}]",
        f"    crm -->|wave 1| customer[{_label(node('CustomerContextAgent'))}]",
        f"    trigger --> store1[({_label('Orchestration store: customer context')})]",
        "    customer --> store1",
        f"    store1 -. {_label('deallocate wave 1')} .-> freed1(({_label('agents released')}))",
        f"    store1 --> product[{_label('Product / SKU system')}]",
        f"    product -->|wave 2| sku[{_label(node('SkuDiscoveryAgent'))}]",
        f"    product -->|wave 2| fit[{_label(node('ProductFitAgent'))}]",
        f"    sku --> store2[({_label('Orchestration store: product context')})]",
        "    fit --> store2",
        f"    store2 -. {_label('deallocate wave 2')} .-> freed2(({_label('agents released')}))",
        f"    store2 --> pricingsys[{_label('Pricing / finance / legal system')}]",
        f"    pricingsys -->|wave 3| pricing[{_label(node('PricingTermsAgent'))}]",
        f"    pricing --> generation[{_label(node('QuoteGenerationAgent'))}]",
        f"    generation --> quote[/{_label('Final quote package')}/]",
    ]
    return "\n".join(lines)


def _label(value: str) -> str:
    return value.replace('"', "'")


def _mermaid_image_url(mermaid: str) -> str:
    encoded = base64.urlsafe_b64encode(mermaid.encode("utf-8")).decode("ascii").rstrip("=")
    return f"https://mermaid.ink/img/{encoded}"


flow_diagram = display_scenario_flow(SCENARIO)
print(flow_diagram.mermaid)


### Read the run output

**Supporting code.** Utilities that turn framework run events into readable text -- the only Agent Framework touchpoints are `result.get_outputs()` / `get_intermediate_outputs()`; the rest is parsing.


In [ ]:
from collections.abc import Mapping, Sequence


def workflow_result_to_text(result: Any) -> str:
    outputs = result.get_outputs() if hasattr(result, "get_outputs") else result
    intermediate = result.get_intermediate_outputs() if hasattr(result, "get_intermediate_outputs") else []
    if not outputs:
        intermediate_text = join_readable_outputs(intermediate)
        return clean_workflow_text(intermediate_text) or "No workflow output was produced."
    output_text = join_readable_outputs(outputs)
    if intermediate and should_use_intermediate_outputs(output_text):
        intermediate_text = join_readable_outputs(intermediate)
        if intermediate_text:
            return clean_workflow_text(intermediate_text)
    return clean_workflow_text(output_text) or "No readable workflow text was produced."


def join_readable_outputs(outputs: Any) -> str:
    return "\n\n".join(text for output in outputs if (text := agent_response_to_text(output)))


def should_use_intermediate_outputs(output_text: str) -> bool:
    normalized = output_text.strip().lower()
    if not normalized:
        return True
    if len(normalized) >= 160:
        return False
    markers = (
        "termination condition",
        "maximum reset count",
        "maximum stall count",
        "workflow terminated",
        "group chat has reached its termination condition",
    )
    return any(marker in normalized for marker in markers)


def agent_response_to_text(value: Any) -> str:
    text = clean_workflow_text(extract_text(value))
    return text


def clean_workflow_text(text: str) -> str:
    """Remove leading framework status lines when useful scenario text follows."""

    lines = text.splitlines()
    while lines and is_framework_status_line(lines[0]) and any(line.strip() for line in lines[1:]):
        lines.pop(0)
        while lines and not lines[0].strip():
            lines.pop(0)
    return "\n".join(lines).strip()


def is_framework_status_line(line: str) -> bool:
    normalized = line.strip().lower()
    return (
        normalized.startswith("invalid next speaker:")
        or normalized.startswith("magentic orchestrator:")
        or normalized.startswith("maximum consecutive function call errors reached")
    )


def extract_text(value: Any, seen: set[int] | None = None) -> str:
    if value is None:
        return ""
    if seen is None:
        seen = set()
    value_id = id(value)
    if value_id in seen:
        return ""
    seen.add(value_id)
    if isinstance(value, str):
        return "" if is_object_repr(value) else value
    text = getattr(value, "text", None)
    if isinstance(text, str) and text and not is_object_repr(text):
        return text
    messages = getattr(value, "messages", None)
    if messages:
        parts: list[str] = []
        for message in messages:
            author = getattr(message, "author_name", None) or getattr(message, "role", None) or "assistant"
            message_text = extract_text(message, seen)
            if message_text:
                parts.append(f"[{author}] {message_text}")
        if parts:
            return "\n".join(parts)
    contents = getattr(value, "contents", None)
    if contents:
        return "\n".join(part for content in contents if (part := extract_text(content, seen)))
    items = getattr(value, "items", None)
    if items and not callable(items):
        return "\n".join(part for item in items if (part := extract_text(item, seen)))
    result = getattr(value, "result", None)
    if result is not None:
        return extract_text(result, seen)
    if isinstance(value, Mapping):
        parts = [extract_text(value.get(key), seen) for key in ("text", "content", "message", "summary", "result")]
        return "\n".join(part for part in parts if part)
    if isinstance(value, Sequence) and not isinstance(value, (bytes, bytearray, str)):
        return "\n".join(part for item in value if (part := extract_text(item, seen)))
    fallback = str(value)
    return "" if is_object_repr(fallback) else fallback


def is_object_repr(value: str) -> bool:
    return value.startswith("<") and " object at 0x" in value and value.endswith(">")




print("Result formatting ready: workflow_result_to_text(...) turns framework events "
      "into readable text.")


## Live Run

The request fans out to the parallel lanes simultaneously. A fan-in executor waits for every response and labels each contribution. Without a synthesizer the labelled findings are the output; with one, a `ConcurrentSynthesisGateExecutor` forwards them to the synthesis agent, which produces the final deliverable. Execution order inside the fan-out is non-deterministic.

> **Instructor note:** `qwen3:14b` runs with `think: False` by default (extended reasoning off).
> Set `OLLAMA_THINK=true` before the environment cell to enable chain-of-thought reasoning --
> useful when debugging unexpected routing decisions or tool call sequences.


In [ ]:
import io
from contextlib import redirect_stderr, redirect_stdout


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
_framework_logs = io.StringIO()
with redirect_stdout(_framework_logs), redirect_stderr(_framework_logs):
    result = await workflow.run(SAMPLE_PROMPT)
framework_logs = _framework_logs.getvalue()
output_text = workflow_result_to_text(result)
if SCENARIO.pattern == "group-chat" and should_use_intermediate_outputs(output_text):
    output_text = group_chat_learning_summary(SCENARIO, SAMPLE_PROMPT, output_text)

if not output_text.strip():
    raise RuntimeError("Workflow completed but produced no readable text.")

render_transcript(output_text)


## What to Inspect

Check that each labelled lane contribution is non-overlapping. Because lanes receive the same input and run independently, their findings should be additive, not redundant. When the scenario declares a synthesizer, confirm its final entry actually reconciles the labelled findings above it rather than repeating one lane.

> **Scenario spotlight:** The payload sets a 150k USD budget cap and a two-week deadline -- finance and operations should engage those constraints, not restate generic risk.


## Experiments

- Raise the budget cap above the vendor's cost and compare the finance lane's verdict.
- Change `RESPONSES_PAYLOAD['input']` and rerun the live cell.
- Override `OLLAMA_MODEL` or `OLLAMA_HOST` before the environment cell to target a different local Ollama setup.
- Inspect `agent_capability_map(SCENARIO)` and tighten one agent's instructions to see how orchestration behavior changes.
- Lower `MAX_TOKENS` for faster runs or raise it when concurrent needs more room.
